# Experiment: Explore the analysis cohorts

Objective:
- State the question you want to answer.
- Define the success criteria.


Compare coverage of the text and supervised cohorts. Use aggregate diagnostics to design the evaluation before fitting models.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from corpus import CLEAN_PATH

movies = pd.read_csv(CLEAN_PATH, parse_dates=["release_date"])
labeled = movies.dropna(subset=["revenue", "release_date"]).copy()
for name, cohort in {"Text corpus": movies, "Supervised cohort": labeled}.items():
    print(f"{name}: {len(cohort):,} movies")
    print("Overview words:", cohort["overview"].str.split().str.len().quantile([0.1, 0.5, 0.9]).to_dict())
    print("Budget coverage:", round(cohort["budget"].notna().mean(), 3))
    print("Exact repeated overview rows:", cohort["overview"].str.casefold().duplicated(keep=False).sum())
print("Positive revenue quantiles:", labeled["revenue"].quantile([0.1, 0.5, 0.9, 0.99]).to_dict())

In [ ]:
coverage = (
    movies.assign(
        release_year=movies["release_date"].dt.year,
        has_revenue=movies["revenue"].notna(),
    )
    .groupby("release_year")["has_revenue"]
    .agg(movies="size", labeled="sum")
)
coverage.plot(y=["movies", "labeled"], ylabel="Movies", xlabel="Release year")
plt.show()
display(coverage.tail(15))